# 3D-модель: движущийся электрод – капля – ванна

Полупластина \(y \ge 0\) (симметрия шва при \(y=0\)). На **одной сетке**: расходная проволока, рост и отрыв капли, полёт и удар в ванну со свободной поверхностью; дуга и электрод едут вдоль \(x\).

**Включено**
- VOF металл / газ: свободная поверхность ванны и капля — один интерфейс
- Движение горелки \(x_e(t)=x_0+v_\mathrm{weld}t\); подача \(-v_\mathrm{feed}\) в твёрдой проволоке
- Пинч в металле, связанном с проволокой; потенциал и \(\mathbf{J}\times\mathbf{B}\) в ванне
- Гауссов нагрев и давление дуги на поверхности ванны (центр в \(x_e\)); анодный нагрев на капле; джоулево тепло
- Фазы (линейная \(f_L\)), Дарси в каше, Марангони + CSF

**Не включено**
- Плазма дуги и магнитное поле вне металла
- Сопоставление с экспериментом (грубое демо)

Плотность газа завышена (\(\rho_g \sim \rho_m/50\)) для устойчивости явной схемы.  
В `QUICK` скорость сварки **завышена** (~0.12 m/s), чтобы за 10–15 ms ход электрода был заметен на сетке.

Свойства по умолчанию — **Al 5182** (Zhao & DebRoy, 2001).

## Схема постановки

\(z=0\) — дно пластины; \(y=0\) — плоскость симметрии. Электрод — вертикальный цилиндр с осью \((x_e,0)\).

```
        y=0 │  wire feed ↓ v_feed, travel → v_weld
            │     ████  R_wire @ x=x_e(t)
            │     ████
            │      ██   ← hanging drop (pinch)
            │       ◯     flight after detachment
   z=H_plate│  ~~~~~~~~~~~~  free surface (VOF)
            │  ████████████  plate / weld pool
        z=0 │────────────────────────── x → Lx
            φ=0 (ground)   arc heat / pressure centred at x_e
```

| Символ | Смысл | Типично (QUICK) |
|---|---|---|
| \(x,y,z\) | вдоль шва, полуширина, высота | 10×3×7 mm |
| \(R_\mathrm{wire}\) | радиус проволоки | 0.6 mm |
| \(I\) | ток | ~170 A |
| \(v_\mathrm{weld}\) | скорость сварки (демо) | ~0.12 m/s |
| \(v_\mathrm{feed}\) | подача | ~4 cm/s |

## Уравнения

Одножидкостный 3D VOF:

$$
\partial_t\alpha + \nabla\cdot(\mathbf{u}\alpha)=0,
\qquad
\nabla\cdot\mathbf{u}=0
$$

Импульс (переменная плотность, CSF, Дарси, Лоренц, давление дуги):

$$
\rho(\alpha)\,(\partial_t\mathbf{u}+\mathbf{u}\cdot\nabla\mathbf{u})
= -\nabla p + \nabla\cdot(\mu\nabla\mathbf{u})
+ \rho\mathbf{g}
+ \sigma\kappa\nabla\alpha
- A_\mathrm{mush}\mathbf{u}
+ \mathbf{J}\times\mathbf{B}
+ p_\mathrm{arc}\,\mathbf{n}|\nabla\alpha|
$$

**Пинч** в металле проволоки (\(R_m(z)\) относительно оси \(x_e\)):

$$
J_z = \frac{I}{\pi R_m^2},\quad
B_\theta = \frac{\mu_0 I r}{2\pi R_m^2},\quad
\mathbf{F}_r = -J_z B_\theta\,\mathbf{e}_r
$$

**Ванна:** \(\nabla\cdot(\sigma_e\nabla\varphi)=0\), ток входит гауссом через свободную поверхность вокруг \(x_e\), \(\varphi=0\) на дне.

Энергия — эффективная теплоёмкость в каше, гауссов/анодный нагрев на интерфейсе, джоулево \(j^2/\sigma_e\).

Объёмы в диагностике — **полный** эквивалент (удвоенная полупластина).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name != 'heat+flow+em_3d':
    cand = ROOT / 'heat+flow+em_3d'
    if cand.is_dir():
        sys.path.insert(0, str(cand))
        MODEL_DIR = cand
    else:
        sys.path.insert(0, str(ROOT))
        MODEL_DIR = ROOT
else:
    sys.path.insert(0, str(ROOT))
    MODEL_DIR = ROOT

import importlib
import weldpool_em3d

importlib.reload(weldpool_em3d)
from weldpool_em3d import Material, Process, WeldPoolEM3D, quick_process

print('model dir:', MODEL_DIR)

## Материал и параметры

`QUICK = True` — грубая сетка (~40×12×28) и короткий прогон (~12 ms), стартовая капля почти критическая.  
`QUICK = False` — мельче шаг и дольше; всё ещё демо, не паспорт дуги.

**Замечание:** `v_weld ≈ 0.12 m/s` в QUICK намеренно велика — при реальных 5–15 mm/s за один цикл капли электрод почти не сдвинется на этой сетке.

In [ ]:
QUICK = True

mat = Material()

if QUICK:
    proc = quick_process(
        enable_flow=True,
        enable_em=True,
        nx=40,
        ny=12,
        nz=28,
        t_end=0.012,
        I_arc=170.0,
        R_drop0=0.00115,
        v_weld=0.12,
        dt_max=4.0e-5,
        u_clip=4.0,
    )
else:
    proc = Process(
        enable_flow=True,
        enable_em=True,
        nx=50,
        ny=16,
        nz=36,
        t_end=0.020,
        R_drop0=0.0009,
        v_weld=0.08,
        dt_max=3.0e-5,
    )

model = WeldPoolEM3D(material=mat, process=proc)
print(
    f'grid {proc.nx}×{proc.ny}×{proc.nz}, '
    f'domain {1e3*proc.Lx:.1f}×{1e3*proc.Ly:.1f}×{1e3*proc.Lz:.1f} mm, '
    f'I={proc.I_arc:.0f} A, v_weld={1e3*proc.v_weld:.0f} mm/s, '
    f'xe0={1e3*model.electrode_x():.2f} mm, '
    f'V0={model.metal_volume()*1e9:.2f} mm³ (full equiv.)'
)

## Начальная геометрия (срез \(y\approx 0\))

In [ ]:
x_mm, z_mm, a0, T0 = model.slice_y0()
Xg, Zg = np.meshgrid(x_mm, z_mm, indexing='ij')
xe0_mm = 1e3 * model.electrode_x()

fig, axs = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
im0 = axs[0].pcolormesh(Xg, Zg, a0, shading='auto', cmap='Blues', vmin=0, vmax=1)
axs[0].contour(Xg, Zg, a0, levels=[0.5], colors='k', linewidths=0.6)
axs[0].axvline(xe0_mm, color='C3', ls='--', lw=0.8, label=r'$x_e$')
axs[0].set_aspect('equal')
axs[0].set_xlabel('x [mm]')
axs[0].set_ylabel('z [mm]')
axs[0].set_title(r'$\alpha$ (y≈0)')
axs[0].legend(fontsize=8)
fig.colorbar(im0, ax=axs[0], fraction=0.046)

im1 = axs[1].pcolormesh(Xg, Zg, T0, shading='auto', cmap='inferno', vmin=300, vmax=max(1500, T0.max()))
axs[1].contour(Xg, Zg, a0, levels=[0.5], colors='cyan', linewidths=0.6)
axs[1].axvline(xe0_mm, color='w', ls='--', lw=0.8)
axs[1].set_aspect('equal')
axs[1].set_xlabel('x [mm]')
axs[1].set_ylabel('z [mm]')
axs[1].set_title('T [K] (y≈0)')
fig.colorbar(im1, ax=axs[1], fraction=0.046)
plt.show()

print(f'R_neck0 = {1e3*model.neck_radius():.3f} mm, components={model._n_components}')

## Расчёт

Полный QUICK на ~13k ячеек может занять несколько минут (проекция с переменной плотностью на каждом шаге).

In [ ]:
diag = model.run(record_every=0.001, progress=True, record_fields=True)
print(
    f'final: t={1e3*model.t:.2f} ms, xe={1e3*model.electrode_x():.2f} mm, '
    f'Tmax={model.T.max():.0f} K, '
    f'Rneck={1e3*model.neck_radius():.3f} mm, '
    f'comp={model._n_components}, detached={model._detached}, '
    f'vol bal={model.volume_balance_ratio():.3f}'
)
print(f'frames: {len(diag.alpha_frames)}')

## Диагностика: шейка, ход электрода, объём, отрыв

In [ ]:
t_ms = 1e3 * np.array(diag.time)

fig, axs = plt.subplots(2, 2, figsize=(10, 6), constrained_layout=True)

axs[0, 0].plot(t_ms, 1e3 * np.array(diag.neck_radius), 'C0')
axs[0, 0].set_ylabel(r'$R_\mathrm{neck}$ [mm]')
axs[0, 0].set_xlabel('t [ms]')
axs[0, 0].set_title('Neck radius')
axs[0, 0].grid(True, alpha=0.3)

axs[0, 1].plot(t_ms, 1e3 * np.array(diag.electrode_x), 'C3', label=r'$x_e(t)$')
axs[0, 1].plot(t_ms, np.array(diag.n_components), 'C1', label='components')
axs[0, 1].set_xlabel('t [ms]')
axs[0, 1].legend(fontsize=8)
axs[0, 1].set_title('Electrode travel & connectivity')
axs[0, 1].grid(True, alpha=0.3)

V0 = diag.metal_volume[0]
axs[1, 0].plot(t_ms, 1e9 * (np.array(diag.metal_volume) - V0), label=r'$\Delta V_\mathrm{metal}$')
axs[1, 0].plot(t_ms, 1e9 * np.array(diag.wire_volume), '--', label=r'$V_\mathrm{wire}$')
axs[1, 0].plot(t_ms, 1e9 * np.array(diag.pool_volume), ':', label=r'$V_\mathrm{pool}$')
axs[1, 0].set_ylabel('volume [mm³] (full equiv.)')
axs[1, 0].set_xlabel('t [ms]')
axs[1, 0].legend(fontsize=8)
axs[1, 0].set_title('Metal volume')
axs[1, 0].grid(True, alpha=0.3)

axs[1, 1].plot(t_ms, diag.T_max, label=r'$T_\mathrm{max}$')
axs[1, 1].plot(t_ms, diag.max_speed, label=r'$|u|_\mathrm{max}$')
axs[1, 1].set_xlabel('t [ms]')
axs[1, 1].legend(fontsize=8)
axs[1, 1].set_title(r'$T_\mathrm{max}$ [K] and speed [m/s]')
axs[1, 1].grid(True, alpha=0.3)

plt.show()

det_times = [1e3 * tt for tt, d in zip(diag.time, diag.detached) if d]
if det_times:
    print(f'first detachment flagged at t ≈ {det_times[0]:.2f} ms')
else:
    print('no detachment in this run — increase I_arc / R_drop0 or t_end')
print(f'Δx_electrode = {1e3*(diag.electrode_x[-1]-diag.electrode_x[0]):.2f} mm')

## Кадры $\alpha$ и $T$ (срез $y\approx 0$)

In [ ]:
n_show = min(6, len(diag.alpha_frames))
idxs = np.linspace(0, len(diag.alpha_frames) - 1, n_show, dtype=int)

fig, axs = plt.subplots(2, n_show, figsize=(2.4 * n_show, 6), constrained_layout=True)
if n_show == 1:
    axs = np.array([[axs[0]], [axs[1]]])

for col, k in enumerate(idxs):
    a_f = diag.alpha_frames[k]
    T_f = diag.T_frames[k]
    xe_k = 1e3 * diag.xe_frames[k]
    t_k = 1e3 * diag.time_frames[k]

    axs[0, col].pcolormesh(Xg, Zg, a_f, shading='auto', cmap='Blues', vmin=0, vmax=1)
    axs[0, col].contour(Xg, Zg, a_f, levels=[0.5], colors='k', linewidths=0.6)
    axs[0, col].axvline(xe_k, color='C3', ls='--', lw=0.7)
    axs[0, col].set_aspect('equal')
    axs[0, col].set_title(f'α  t={t_k:.1f} ms', fontsize=9)
    if col == 0:
        axs[0, col].set_ylabel('z [mm]')

    im = axs[1, col].pcolormesh(
        Xg, Zg, T_f, shading='auto', cmap='inferno',
        vmin=300, vmax=max(1500, model.T.max()),
    )
    axs[1, col].contour(Xg, Zg, a_f, levels=[0.5], colors='cyan', linewidths=0.6)
    axs[1, col].axvline(xe_k, color='w', ls='--', lw=0.7)
    axs[1, col].set_aspect('equal')
    axs[1, col].set_xlabel('x [mm]')
    if col == 0:
        axs[1, col].set_ylabel('z [mm]')

plt.colorbar(im, ax=axs[1, :].tolist(), fraction=0.02, pad=0.02, label='T [K]')
plt.show()

## Анимация по времени (срез $y\approx 0$)

Нужен прогон с `record_fields=True`. Красная пунктирная линия — текущее \(x_e\).

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

assert diag.alpha_frames, 'No field frames — re-run with record_fields=True'

t_fr = np.array(diag.time_frames)
alpha_fr = diag.alpha_frames
T_fr = diag.T_frames
xe_fr = np.array(diag.xe_frames)

T_vmin = float(min(f.min() for f in T_fr))
T_vmax = float(max(max(f.max() for f in T_fr), mat.T_liquidus + 50.0))

fig, axs = plt.subplots(1, 2, figsize=(10, 4.5), constrained_layout=True)

im_a = axs[0].pcolormesh(Xg, Zg, alpha_fr[0], shading='auto', cmap='Blues', vmin=0, vmax=1)
line_a = axs[0].axvline(1e3 * xe_fr[0], color='C3', ls='--', lw=0.9)
fig.colorbar(im_a, ax=axs[0], fraction=0.046, label=r'$\alpha$')
axs[0].set_aspect('equal')
axs[0].set_xlabel('x [mm]')
axs[0].set_ylabel('z [mm]')
title_a = axs[0].set_title(f'α  t={1e3*t_fr[0]:.2f} ms')

im_T = axs[1].pcolormesh(Xg, Zg, T_fr[0], shading='auto', cmap='inferno', vmin=T_vmin, vmax=T_vmax)
line_T = axs[1].axvline(1e3 * xe_fr[0], color='w', ls='--', lw=0.9)
fig.colorbar(im_T, ax=axs[1], fraction=0.046, label='T [K]')
axs[1].set_aspect('equal')
axs[1].set_xlabel('x [mm]')
axs[1].set_ylabel('z [mm]')
title_T = axs[1].set_title(f'T  t={1e3*t_fr[0]:.2f} ms')


def _clear_contours(ax, keep):
    for coll in list(ax.collections):
        if coll not in keep:
            try:
                coll.remove()
            except ValueError:
                pass


def update_alpha_T(i):
    a_i = alpha_fr[i]
    T_i = T_fr[i]
    xe_i = 1e3 * xe_fr[i]
    im_a.set_array(a_i.ravel())
    im_T.set_array(T_i.ravel())
    line_a.set_xdata([xe_i, xe_i])
    line_T.set_xdata([xe_i, xe_i])
    title_a.set_text(f'α  t={1e3*t_fr[i]:.2f} ms')
    title_T.set_text(f'T  t={1e3*t_fr[i]:.2f} ms')
    _clear_contours(axs[0], {im_a})
    _clear_contours(axs[1], {im_T})
    axs[0].contour(Xg, Zg, a_i, levels=[0.5], colors='k', linewidths=0.8)
    axs[1].contour(Xg, Zg, a_i, levels=[0.5], colors='cyan', linewidths=0.8)
    return (im_a, im_T, title_a, title_T, line_a, line_T)


anim_alpha_T = FuncAnimation(fig, update_alpha_T, frames=len(alpha_fr), interval=180, blit=False)
plt.close(fig)
HTML(anim_alpha_T.to_jshtml())